In [1]:
import xtrack as xt
import numpy as np
import sys
helpers_path = f'../'
sys.path.insert(0, helpers_path)
from helpers_for_imperfections_model import generate_monitor_misalignments, apply_monitor_misalignments, create_elements_switch, apply_orbit_correction
from helpers_from_xutil import match_tune_chroma

In [2]:
# CHOOSE SEED HERE
seed = 4

In [3]:
line_version = "LCC_106-2-3_z"

# Load reference line with correctors installed (note that we cycle it to the rf cavity)
line = xt.Line.from_json('lattices/reference_lattice_LCC_V106/line_fccee_p_ring_LCC_106-2-3_z_merged_dipoles_with_correctors.json')
line.cycle(name_first_element='rf400', inplace=True) # cycle to rf cavity
line.configure_radiation(model=None, model_beamstrahlung=None) # disable radiation 
line.twiss_default['method'] = '4d' # switch to 4d Twiss method
print(f'Loaded the {line_version} line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.')

tt = line.get_table()
tw_ref = line.twiss() #tw_ref has 4D Twiss

Loading line from dict: 100%|██████████| 15545/15545 [00:03<00:00, 4608.95it/s]


Done loading line from dict.           
Loaded the LCC_106-2-3_z line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.


In [4]:
# Load line with imperfections switches installed
line = xt.Line.from_json(f'lattices/lattices_with_imperfections/{line_version}_line_with_imperfections_switches_seed{seed}.json')
# Remember that the line is cycled to rf cavity, has radiation disabled, and has Twiss method set to 4D

Loading line from dict: 100%|██████████| 15545/15545 [00:02<00:00, 5750.61it/s]


Done loading line from dict.           


In [5]:
# ARCS
arc_markers = [
    ('end_ds_start_arc_ipa', 'end_arc_start_ds_ipb'),
    ('end_ds_start_arc_ipb', 'end_arc_start_ds_ipd'),
    ('end_ds_start_arc_ipd', 'end_arc_start_ds_ipf'),
    ('end_ds_start_arc_ipf', 'end_arc_start_ds_ipg'),
    ('end_ds_start_arc_ipg', 'end_arc_start_ds_iph'),
    ('end_ds_start_arc_iph', 'end_arc_start_ds_ipj'),
    ('end_ds_start_arc_ipj', 'end_arc_start_ds_ipl'),
    ('end_ds_start_arc_ipl', 'end_arc_start_ds_ipa')]

# DISPERSION SUPPRESSION REGIONS (count as part of the arc)
DS_markers = [
    ('end_straight_start_ds_ipa', 'end_ds_start_arc_ipa'),
    ('end_arc_start_ds_ipb', 'end_ds_start_straight_ipb'),
    ('end_straight_start_ds_ipb', 'end_ds_start_arc_ipb'),
    ('end_arc_start_ds_ipd', 'end_ds_start_straight_ipd'),
    ('end_straight_start_ds_ipd', 'end_ds_start_arc_ipd'),
    ('end_arc_start_ds_ipf', 'end_ds_start_straight_ipf'),
    ('end_straight_start_ds_ipf', 'end_ds_start_arc_ipf'),
    ('end_arc_start_ds_ipg', 'end_ds_start_straight_ipg'),
    ('end_straight_start_ds_ipg', 'end_ds_start_arc_ipg'),
    ('end_arc_start_ds_iph', 'end_ds_start_straight_iph'),
    ('end_straight_start_ds_iph', 'end_ds_start_arc_iph'),
    ('end_arc_start_ds_ipj', 'end_ds_start_straight_ipj'),
    ('end_straight_start_ds_ipj', 'end_ds_start_arc_ipj'),
    ('end_arc_start_ds_ipl', 'end_ds_start_straight_ipl'),
    ('end_straight_start_ds_ipl', 'end_ds_start_arc_ipl'),
    ('end_arc_start_ds_ipa', 'end_ds_start_straight_ipa')]

# STRAIGHT SECTIONS
straight_markers = [
    ('end_ds_start_straight_ipb', 'end_straight_start_ds_ipb'),
    ('end_ds_start_straight_ipd', 'end_straight_start_ds_ipd'),
    ('end_ds_start_straight_ipf', 'end_straight_start_ds_ipf'),
    ('end_ds_start_straight_ipg', 'end_straight_start_ds_ipg'),
    ('end_ds_start_straight_iph', 'end_straight_start_ds_iph'),
    ('end_ds_start_straight_ipj', 'end_straight_start_ds_ipj'),
    ('end_ds_start_straight_ipl', 'end_straight_start_ds_ipl'),
    ('end_ds_start_straight_ipa', 'end_straight_start_ds_ipa')]

# Split the straight markers into INTERACTION REGION (IR) and TECHNICAL REGION (TR) markers
IR_markers = [
    ('end_ds_start_straight_ipa', 'end_straight_start_ds_ipa'),
    ('end_ds_start_straight_ipd', 'end_straight_start_ds_ipd'),
    ('end_ds_start_straight_ipg', 'end_straight_start_ds_ipg'),
    ('end_ds_start_straight_ipj', 'end_straight_start_ds_ipj')]

TR_markers = [
    ('end_ds_start_straight_ipb', 'end_straight_start_ds_ipb'),
    ('end_ds_start_straight_ipf', 'end_straight_start_ds_ipf'),
    ('end_ds_start_straight_iph', 'end_straight_start_ds_iph'),
    ('end_ds_start_straight_ipl', 'end_straight_start_ds_ipl')]

### Prepare the line for the correction procedure

In [6]:
# Generate the monitor misalignment dictionary

# BPMs inherit the imperfections of the quadrupoles they are attached to, which themselves may inherit the imperfections of the girders they are mounted on
# Therefore, remember to set all the quadrupole & girder misalignment switches to 1 before calling the function
line.vars['on_misalignment_quad_arc'] = 1
line.vars['on_misalignment_quad_fd'] = 1
line.vars['on_misalignment_quad_ff'] = 1
line.vars['on_misalignment_quad_tr'] = 1
line.vars['on_misalignment_girder'] = 1
monitor_alignment = generate_monitor_misalignments(line, pattern='bpm', attrs=['shift_x', 'shift_y', 'rot_s_rad'], 
                                                    line_table=tt, element_type='Marker')
line.vars['on_misalignment_quad_arc'] = 0
line.vars['on_misalignment_quad_fd'] = 0
line.vars['on_misalignment_quad_ff'] = 0
line.vars['on_misalignment_quad_tr'] = 0
line.vars['on_misalignment_girder'] = 0

In [7]:
# Apply additional imperfections to the BPMS (i.e. position resolution errors)
bpm_dict = apply_monitor_misalignments(monitor_alignment, seed, sigmas=[10e-6, 10e-6, 100e-6])

In [8]:
# Set switches for powering the sextupoles by using regex/marker-based filtering (this is needed to ramp up the sextupole strengths during the correction routine)
# and then deactivate the sextupoles

# ARC SEXTUPOLES
create_elements_switch(line, line_table=tt, switch_name='on_powering_sext_arc', marker_pairs=[arc_markers, DS_markers], element_type='Sextupole')
line.vars['on_powering_sext_arc'] = 0

# INTERACTION REGION (IR) SEXTUPOLES 8excluding crab9
create_elements_switch(line, line_table=tt, switch_name='on_powering_sext_IR', marker_pairs=[IR_markers], element_type='Sextupole', except_pattern=['scrab'])
line.vars['on_powering_sext_IR'] = 0

# CRAB SEXTUPOLES ('scrab')
create_elements_switch(line, line_table=tt, switch_name='on_powering_sext_crab', element_type='Sextupole', pattern='scrab')
line.vars['on_powering_sext_crab'] = 0

print('Switches for powering the sextupoles created successfully!')

1760 elements found for switch on_powering_sext_arc
160 elements found for switch on_powering_sext_IR
32 elements found for switch on_powering_sext_crab
Switches for powering the sextupoles created successfully!


In [9]:
# Load the steering correctors and monitors required for the correction procedure
steering_monitors=tt.rows['bpm.*']
line.steering_monitors_x = steering_monitors.name
line.steering_monitors_y = steering_monitors.name
print(f'Found', len(line.steering_monitors_x), 'BPMs in the line.')

line.steering_correctors_x = tt.rows['hcor.*'].name
line.steering_correctors_y = tt.rows['vcor.*'].name
print(f'Found', len(line.steering_correctors_x), 'horizontal correctors in the line.')
print(f'Found', len(line.steering_correctors_y), 'vertical correctors in the line.')

Found 2639 BPMs in the line.
Found 2679 horizontal correctors in the line.
Found 2679 vertical correctors in the line.


In [10]:
# Set compute chromatic properties to False
line.twiss_default['compute_chromatic_properties'] = False
print('Switched to compute_chromatic_properties=False.')

Switched to compute_chromatic_properties=False.


### Switch on all the imperfections

In [11]:
# ARCS
line.vars['on_misalignment_dip_arc'] = 1
line.vars['on_misalignment_quad_arc'] = 1
line.vars['on_misalignment_sext_arc'] = 1

line.vars['on_field_error_dip_arc'] = 1
line.vars['on_field_error_quad_arc'] = 1
line.vars['on_field_error_sext_arc'] = 1

# STRAIGHTS
line.vars['on_misalignment_dip_ir'] = 1
line.vars['on_misalignment_dip_tr'] = 1
line.vars['on_misalignment_quad_fd'] = 1
line.vars['on_misalignment_quad_ff'] = 1
line.vars['on_misalignment_quad_tr'] = 1
line.vars['on_misalignment_sext_ir'] = 1

line.vars['on_field_error_dip_ir'] = 1
line.vars['on_field_error_dip_tr'] = 1
line.vars['on_field_error_quad_fd'] = 1
line.vars['on_field_error_quad_ff'] = 1
line.vars['on_field_error_quad_tr'] = 1
line.vars['on_field_error_sext_ir'] = 1

# GIRDERS
line.vars['on_misalignment_girder'] = 1

print('All imperfections applied.')

All imperfections applied.


### Run the orbit correction routine

In [12]:
# Choose a corrector strength limit if desired
corr_limit = 5e-4
# Choose the number of singular values to be used for the correction
num_sing_vals = 2500

#### Initial orbit correction

In [24]:
# Initial orbit correction (defaults to threading almost always)

trial_line = line.copy()
try: 
    apply_orbit_correction(trial_line, tw_ref, monitor_alignment, num_sing_vals=num_sing_vals, corr_limit=corr_limit)
    print(f'Initial orbit correction completed successfully!')
    line = trial_line
except Exception:
    # In few cases, the initial orbit correction may fail due to the strong perturbations induced by the applied imperfections.
    # If keen to push a particular seed through the correction routine nonetheless, can try to take monitor alignment out of the argument of the orbit correction algorithm
    print(f'Initial orbit correction failed.')
    print('Retrying without monitor alignment as argument...')
    trial_line = line.copy()
    apply_orbit_correction(trial_line, tw_ref, num_sing_vals=num_sing_vals, corr_limit=corr_limit)
    line = trial_line



Warning! Need second attempt on closed orbit search


Starting orbit correction with threading method...
Stop at s=5000, global rms = [x: 2.23e-03 -> 6.37e-06, y: 1.23e-03 -> 9.14e-06]
Stop at s=10000, global rms = [x: 2.06e-03 -> 6.37e-06, y: 1.73e-03 -> 6.46e-06]
Stop at s=15000, global rms = [x: 3.33e-03 -> 1.43e-05, y: 4.40e-03 -> 5.85e-06]
Stop at s=20000, global rms = [x: 8.41e-04 -> 2.82e-06, y: 1.02e-03 -> 3.82e-06]
Stop at s=25000, global rms = [x: 3.33e-04 -> 2.95e-06, y: 1.12e-03 -> 2.70e-06]
Stop at s=30000, global rms = [x: 1.05e-03 -> 5.14e-06, y: 9.65e-04 -> 4.12e-06]
Stop at s=35000, global rms = [x: 1.06e-03 -> 1.34e-04, y: 3.31e-03 -> 1.59e-04]
Stop at s=40000, global rms = [x: 1.25e-03 -> 8.04e-06, y: 4.38e-03 -> 1.18e-05]
Stop at s=45000, global rms = [x: 4.79e-04 -> 7.07e-06, y: 8.35e-04 -> 2.96e-06]
Stop at s=50000, global rms = [x: 5.90e-04 -> 1.73e-06, y: 3.20e-04 -> 1.96e-06]
Stop at s=55000, global rms = [x: 6.25e-04 -> 2.13e-06, y: 9.56e-04 -> 5.09e-06]
Sto

#### Initial tune matching

In [25]:
# Trick is to use try/except for the tune matching as the optics may be so distorted that tune matching is not yet possible at this stage
try: 
    match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
    print(f'Initial tune matching completed successfully!')
except Exception as e:
    print(f'Initial tune matching failed due to: {e}')

                                             
Optimize - start penalty: 0.04935                           
Matching: model call n. 11 penalty = 1.5438e-07              
Optimize - end penalty:  1.54382e-07                            
Target status:               nalty = 1.5438e-07              
id state tag  tol_met       residue   current_val    target_val description                           
0  ON    tune    True   1.07391e-08        194.16        194.16 'qx', val=194.16, tol=1e-05, weight=10
1  ON    tune    True   -1.1091e-08         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10 
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00926905 None           0.00926927         1e-08             1
1  ON    quad OK  kqd1 None           -0.0136955 None           -0.0136956         1e-08             1
Initial tune matching completed successfully!


#### Sextupole ramping

In [26]:
# Arc sextupole ramping
for i in [0.33, 0.66, 0.90, 1]: # choose suitable iteration steps for the sextupole ramping here
    print(f'Ramping up arc sextupoles to {i*100}%...')
    line.vars['on_powering_sext_arc'] = i

    apply_orbit_correction(line, tw_ref, monitor_alignment, num_sing_vals=num_sing_vals, corr_limit=corr_limit)
    print(f'Orbit corrected successfully for arc sextupole ramping up to {i*100}% !')

    try:
        match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
        print(f'Tune matched successfully after orbit correction with arc sextupole ramping up to {i*100}% !')
    except Exception as e:
        print(f'Tune matching failed at i={i*100}% during arc sextupole ramping due to: {e}')

Ramping up arc sextupoles to 33.0%...
Iteration 0, x_rms: 2.59e-05 -> 2.59e-05, y_rms: 3.82e-05 -> 3.36e-05
Iteration 1, x_rms: 2.59e-05 -> 2.59e-05, y_rms: 3.36e-05 -> 3.36e-05
Orbit corrected successfully for arc sextupole ramping up to 33.0% !
                                             
Optimize - start penalty: 0.02646                           
Matching: model call n. 11 penalty = 7.5174e-08              
Optimize - end penalty:  7.51742e-08                            
Target status:               nalty = 7.5174e-08              
id state tag  tol_met       residue   current_val    target_val description                           
0  ON    tune    True   5.26711e-09        194.16        194.16 'qx', val=194.16, tol=1e-05, weight=10
1  ON    tune    True  -5.36369e-09         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10 
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad O

In [27]:
# IR sextupole ramping
for i in [0.25, 0.50, 0.75, 0.85, 0.90, 0.95, 1]: # choose suitable iteration steps for the sextupole ramping here
    print(f'Ramping up IR sextupoles to {i*100}%...')
    line.vars['on_powering_sext_IR'] = i

    apply_orbit_correction(line, tw_ref, monitor_alignment, num_sing_vals=num_sing_vals, corr_limit=corr_limit)
    print(f'Orbit corrected successfully for IR sextupole ramping up to {i*100}% !')
    
    try: 
        match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
        print(f'Tune matched successfully after orbit correction with IR sextupole ramping up to {i*100}% !')
    except Exception as e:
        print(f'Tune matching failed at i={i*100}% during IR sextupole ramping due to: {e}')

Ramping up IR sextupoles to 25.0%...
Iteration 0, x_rms: 2.59e-05 -> 2.59e-05, y_rms: 3.36e-05 -> 3.36e-05
Orbit corrected successfully for IR sextupole ramping up to 25.0% !
                                             
Optimize - start penalty: 0.258                             
Matching: model call n. 11 penalty = 1.4060e-07              
Optimize - end penalty:  1.40599e-07                            
Target status:               nalty = 1.4060e-07              
id state tag  tol_met       residue   current_val    target_val description                           
0  ON    tune    True   9.88271e-09        194.16        194.16 'qx', val=194.16, tol=1e-05, weight=10
1  ON    tune    True  -1.00006e-08         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10 
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00926856 None           0.00926895         1e-0

In [28]:
# Crab sextupole ramping
for i in [0.33, 0.66, 0.90, 1]: # choose suitable iteration steps for the sextupole ramping here
    print(f'Ramping up crab sextupoles to {i*100}%...')
    line.vars['on_powering_sext_crab'] = i

    apply_orbit_correction(line, tw_ref, monitor_alignment, num_sing_vals=num_sing_vals, corr_limit=corr_limit)
    print(f'Orbit corrected successfully for crab sextupole ramping up to {i*100}% !')

    try:
        match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
        print(f'Tune matched successfully after orbit correction with crab sextupole ramping up to {i*100}% !')
    except Exception as e:
        print(f'Tune matching failed at i={i*100}% during crab sextupole ramping due to: {e}')

Ramping up crab sextupoles to 33.0%...
Iteration 0, x_rms: 2.59e-05 -> 2.59e-05, y_rms: 3.36e-05 -> 3.36e-05
Orbit corrected successfully for crab sextupole ramping up to 33.0% !
                                             
Optimize - start penalty: 0.004973                          
Matching: model call n. 6 penalty = 2.2597e-06              
Optimize - end penalty:  2.2597e-06                            
Target status:               alty = 2.2597e-06              
id state tag  tol_met       residue   current_val    target_val description                           
0  ON    tune    True  -2.22964e-07        194.16        194.16 'qx', val=194.16, tol=1e-05, weight=10
1  ON    tune    True   3.67332e-08         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10 
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00926769 None           0.00926768         1e-

In [29]:
# Reintroduce the chromatic properties
line.twiss_default['compute_chromatic_properties'] = True
print('Switched to compute_chromatic_properties=True since all sextupoles are now fully ramped up!')

Switched to compute_chromatic_properties=True since all sextupoles are now fully ramped up!


In [31]:
# Save the orbit-corrected line
line.to_json(f'lattices/lattices_with_corrected_imperfections/01_orbit_corrected_only/{line_version}_line_orbit_corrected_seed{seed}.json')